# PDS4 Validation Utilities

## Overview
This module provides validation and metadata extraction utilities for PDS4 labels using the official pds4_tools library.

Per ADR-008: Integration of PDS4 Python Tools for label validation and metadata access.

## Function Reference

### validate_pds4_label
Validate a PDS4 label against its associated data structure.

Arguments: label_path (str or Path) - Path to PDS4 label file

Returns: dict with keys: valid, warnings, structure

Notes: Uses pds4_tools to read label and compare declared vs. actual data dimensions.

### extract_label_metadata
Extract structured metadata from a PDS4 label.

Arguments: label_path (str or Path) - Path to PDS4 label file  

Returns: dict with processed metadata (level, binning, acquisition params)

Notes: Uses structures.label.to_dict(cast_values=True) for structured access.

### get_csv_field_metadata
Extract CSV column metadata from PDS4 table labels.

Arguments: hk_path (str or Path) - Path to HK data file (searches for matching label)

Returns: pd.DataFrame with columns: name, unit, description, data_type

Notes: Extracts field metadata from PDS4 Table_Delimited section.

### extract_mertis_hk_columns
Extract column names from MERTIS HK files (updated per ADR-008).

Arguments: in_path (Path), use_pds4_tools (bool, default=True)

Returns: list of column names

Notes: Uses pds4tools when available, falls back to xmltodict.

In [ ]:
#| default_exp pds4_validation
#| export
import pathlib
from typing import Union, Optional
import pandas as pd

# Optional import - will fallback if not available
try:
    import pds4_tools
    PDS4_TOOLS_AVAILABLE = True
except ImportError:
    PDS4_TOOLS_AVAILABLE = False
    pds4_tools = None

In [ ]:
#| export
def validate_pds4_label(label_path: Union[str, pathlib.Path]) -> dict:
    """
    Validate a PDS4 label against its associated data structure.
    
    Per ADR-008: Uses pds4_tools to verify label declarations match actual data.
    
    Args:
        label_path: Path to PDS4 label file (.lbl or .xml)
        
    Returns:
        dict with keys:
            - 'valid': bool - Whether label passes validation
            - 'warnings': list - List of warning messages
            - 'structure': dict - Extracted structure information
            
    Raises:
        ImportError: If pds4_tools is not installed
        FileNotFoundError: If label file does not exist
        
    Example:
        >>> from mertisreader import validate_pds4_label
        >>> result = validate_pds4_label('path/to/label.lbl')
        >>> print(f"Valid: {result['valid']}")
        >>> print(f"Warnings: {result['warnings']}")
    """
    if not PDS4_TOOLS_AVAILABLE:
        raise ImportError(
            "pds4_tools is required for label validation. "
            "Install with: pip install pds4_tools>=1.10.0"
        )
    
    label_path = pathlib.Path(label_path)
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")
    
    result = {
        'valid': True,
        'warnings': [],
        'structure': {}
    }
    
    try:
        # Read with lazy_load for memory efficiency (ADR-006 compatibility)
        structures = pds4_tools.read(str(label_path), lazy_load=True)
        
        # Extract label info using to_dict() instead of direct attribute access
        label_dict = structures.label.to_dict(cast_values=True)
        prod_obs = label_dict.get('Product_Observational', {})
        ident_area = prod_obs.get('Identification_Area', {})
        
        result['structure']['label_version'] = ident_area.get('version_id')
        result['structure']['product_class'] = ident_area.get('product_class')
        result['structure']['logical_identifier'] = ident_area.get('logical_identifier')
        
        # Check for each structure and validate
        for i, structure in enumerate(structures):
            struct_type = structure.type if hasattr(structure, 'type') else 'unknown'
            result['structure'][f'structure_{i}'] = {
                'type': struct_type,
                'data_loaded': structure.data_loaded if hasattr(structure, 'data_loaded') else None
            }
            
            # For table structures, check field definitions
            if hasattr(structure, 'fields') and structure.fields:
                result['structure'][f'structure_{i}']['field_count'] = len(structure.fields)
                
    except Exception as e:
        result['valid'] = False
        result['warnings'].append(f"Validation error: {str(e)}")
    
    return result

In [ ]:
#| export
def extract_label_metadata(label_path: Union[str, pathlib.Path]) -> dict:
    """
    Extract structured metadata from a PDS4 label.
    
    Per ADR-008: Uses pds4tools Label.to_dict() for structured metadata access.
    
    Args:
        label_path: Path to PDS4 label file (.lbl or .xml)
        
    Returns:
        dict with extracted metadata including:
            - product_line: PDS product line identifier
            - label_version: PDS4 label version
            - processing_level: MERTIS processing level (RAW/CAL/PAR)
            - acquisition: Acquisition parameters if available
            - instrument: Instrument configuration if available
            
    Raises:
        ImportError: If pds4_tools is not installed
        FileNotFoundError: If label file not found
        
    Example:
        >>> from mertisreader import extract_label_metadata
        >>> meta = extract_label_metadata('path/to/label.lbl')
        >>> print(f"Processing level: {meta.get('processing_level')}")
    """
    if not PDS4_TOOLS_AVAILABLE:
        raise ImportError(
            "pds4_tools is required for metadata extraction. "
            "Install with: pip install pds4_tools>=1.10.0"
        )
    
    label_path = pathlib.Path(label_path)
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")
    
    result = {
        'product_line': None,
        'label_version': None,
        'processing_level': None,
        'acquisition': {},
        'instrument': {}
    }
    
    try:
        structures = pds4_tools.read(str(label_path))
        label_dict = structures.label.to_dict(cast_values=True)
        
        # Extract basic product info
        prod_obs = label_dict.get('Product_Observational', {})
        ident_area = prod_obs.get('Identification_Area', {})
        result['product_line'] = ident_area.get('product_line')
        result['label_version'] = ident_area.get('version_id')
        
        # Extract MERTIS-specific processing level
        obs_area = prod_obs.get('Observation_Area', {})
        primary_summary = obs_area.get('Primary_Result_Summary', {})
        processing_level = primary_summary.get('processing_level')
        if processing_level:
            # Normalize to uppercase (Raw -> RAW, Cal -> CAL, etc.)
            result['processing_level'] = processing_level.upper()
        
        # Extract acquisition info
        result['acquisition'] = {
            'start_time': obs_area.get('Time_Coordinates', {}).get('start_date_time'),
            'stop_time': obs_area.get('Time_Coordinates', {}).get('stop_date_time'),
            'target': obs_area.get('Target_Identification', {}).get('name'),
            'mission_phase': obs_area.get('Mission_Area', {}).get('psa:Mission_Information', {}).get('psa:Mission_Phase', {}).get('psa:name')
        }
        
        # Extract instrument info
        observing_system = obs_area.get('Observing_System', {})
        result['instrument'] = {
            'name': observing_system.get('name'),
            'components': [c.get('name') for c in observing_system.get('Observing_System_Component', [])]
        }
                
    except Exception as e:
        result['error'] = str(e)
    
    return result

In [ ]:
#| export
def get_csv_field_metadata(hk_path: Union[str, pathlib.Path]) -> pd.DataFrame:
    """
    Extract CSV column metadata from PDS4 table labels.
    
    Per ADR-008: Extracts field-level metadata (units, descriptions, types) from
    PDS4 Table_Delimited sections.
    
    Args:
        hk_path: Path to HK data file (.dat). Will search for matching .xml/.lblx label.
        
    Returns:
        pd.DataFrame with columns:
            - name: Field/column name
            - unit: Physical unit (if specified)
            - description: Field description
            - data_type: PDS4 data type (ASCII_String, ASCII_Real, etc.)
            
    Raises:
        ImportError: If pds4_tools is not installed
        FileNotFoundError: If label file not found
        
    Example:
        >>> from mertisreader import get_csv_field_metadata
        >>> meta_df = get_csv_field_metadata('path/to/hk.dat')
        >>> print(meta_df[['name', 'unit', 'description']])
    """
    if not PDS4_TOOLS_AVAILABLE:
        raise ImportError(
            "pds4_tools is required for field metadata extraction. "
            "Install with: pip install pds4_tools>=1.10.0"
        )
    
    hk_path = pathlib.Path(hk_path)
    
    # Find matching label file
    label_path = hk_path.with_suffix('.xml')
    if not label_path.exists():
        label_path = hk_path.with_suffix('.lblx')
    
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found for {hk_path}")
    
    rows = []
    
    try:
        structures = pds4_tools.read(str(label_path))
        
        # Iterate through all structures looking for tables
        for structure in structures:
            # Access fields via structure.fields - this gives the actual data arrays
            # Each field has .meta_data with the field definition
            if hasattr(structure, 'fields') and structure.fields:
                for field_data in structure.fields:
                    if hasattr(field_data, 'meta_data'):
                        meta = field_data.meta_data
                        # meta is a list of (key, value) tuples
                        meta_dict = dict(meta) if meta else {}
                        row = {
                            'name': meta_dict.get('name'),
                            'unit': meta_dict.get('unit'),
                            'description': meta_dict.get('description'),
                            'data_type': meta_dict.get('data_type')
                        }
                        rows.append(row)
                    
    except Exception as e:
        raise RuntimeError(f"Failed to extract field metadata: {str(e)}")
    
    return pd.DataFrame(rows)

In [ ]:
#| export
def extract_mertis_hk_columns(in_path: Union[str, pathlib.Path], use_pds4_tools: bool = True) -> list:
    """
    Extract column names from an MERTIS HK XML/LBLX label file.
    
    Per ADR-008: Updated to use pds4tools when available, with fallback to xmltodict.
    
    Args:
        in_path: Path to HK data file (.dat)
        use_pds4_tools: If True (default), attempt to use pds4tools. Falls back to xmltodict.
        
    Returns:
        list of column names from the HK table definition
        
    Example:
        >>> from mertisreader import extract_mertis_hk_columns
        >>> columns = extract_mertis_hk_columns('path/to/hk.dat')
        >>> print(columns)
    """
    in_path = pathlib.Path(in_path)
    
    # Find matching label file
    label_path = in_path.with_suffix('.xml')
    if not label_path.exists():
        label_path = in_path.with_suffix('.lblx')
    
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found for {in_path}")
    
    # Try pds4tools first if requested
    if use_pds4_tools and PDS4_TOOLS_AVAILABLE:
        try:
            structures = pds4_tools.read(str(label_path))
            for structure in structures:
                if hasattr(structure, 'meta_data') and structure.meta_data:
                    return [field.name for field in structure.meta_data]
        except Exception:
            pass  # Fall through to xmltodict fallback
    
    # Fallback to xmltodict (original implementation)
    import xmltodict
    
    with open(label_path, "rb") as f:
        obj = xmltodict.parse(f, xml_attribs=False)
    
    # Extract column names from parsed XML
    try:
        table_def = obj['Product_Observational']['File_Area_Observational']['Table_Delimited']['Record_Delimited']['Field_Delimited']
        return [x['name'] for x in table_def]
    except (KeyError, TypeError):
        # Try alternative path for different label structure
        try:
            table_def = obj['Product']['File_Area']['Table']['Record']['Field']
            return [x['name'] for x in table_def]
        except (KeyError, TypeError):
            raise ValueError(f"Could not parse table definition from {label_path}")

In [ ]:
#| hide_input
# Test against sample data
import logging
logging.basicConfig(level=logging.ERROR)

# Check if pds4_tools is available
print(f"pds4_tools available: {PDS4_TOOLS_AVAILABLE}")

if PDS4_TOOLS_AVAILABLE:
    # Find a sample FITS file with label
    sample_dir = pathlib.Path("../data/bcmer_tm_all_START-20200409T000000_END-20200410T000000_CRE-20240717T132010-ParamEventBootSciHK-short/raw")
    
    if sample_dir.exists():
        # Look for HK files
        hk_files = list(sample_dir.glob("*_hk_*.dat"))
        fits_files = list(sample_dir.glob("*tis*.fits"))
        
        print(f"\nFound {len(hk_files)} HK files and {len(fits_files)} FITS files")
        
        if hk_files:
            print(f"\nTesting extract_mertis_hk_columns with pds4tools...")
            columns = extract_mertis_hk_columns(hk_files[0], use_pds4_tools=True)
            print(f"Columns from {hk_files[0].name}: {len(columns)} columns")
            print(f"First 5 columns: {columns[:5]}")
        
        if fits_files:
            print(f"\nTesting validate_pds4_label...")
            # For FITS files, the label is embedded, so we would need to extract it
            # For now, just show the function is available
            print(f"validate_pds4_label function ready (requires standalone label file)")
else:
    print("pds4_tools not installed - functions will raise ImportError when called")
    print("Install with: pip install pds4_tools>=1.10.0")

In [ ]:
#| hide_input
# Test backward compatibility - xmltodict fallback
import pathlib

sample_dir = pathlib.Path("../data/bcmer_tm_all_START-20200409T000000_END-20200410T000000_CRE-20240717T132010-ParamEventBootSciHK-short/raw")
if sample_dir.exists():
    hk_files = list(sample_dir.glob("*_hk_*.dat"))
    if hk_files:
        # Test with pds4tools disabled (fallback mode)
        columns_fallback = extract_mertis_hk_columns(hk_files[0], use_pds4_tools=False)
        print(f"Fallback (xmltodict) columns: {len(columns_fallback)} columns")
        print(f"First 5: {columns_fallback[:5]}")